Problem Statement:

An e-commerce company wants to predict customer ratings for Amazon products using product information and customer reviews. Use the available dataset of approximately 1465 records in kaggle.



Tasks:

1. Load the dataset, handle missing values, and preprocess product features and review text.
2. Tokenize and encode customer reviews into numerical sequences.
3. Select and justify a suitable deep learning model for product rating prediction.
4. Train the model and evaluate it using appropriate metrics.
5. Compare actual and predicted ratings and demonstrate predictions on unssen reviews.

**For this task, a Bidirectional Long Short-Term Memory (BiLSTM) model is chosen for the deep learning architecture**

Importing libraries


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

# Task 1: Load Dataset and Preprocess

In [3]:
df = pd.read_csv('amazon.csv')

In [4]:
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

In [5]:
df.dropna(subset=['rating', 'review_content', 'review_title'], inplace=True)

In [6]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text) # Remove HTML
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    return text

In [7]:
df['full_review'] = df['review_title'] + " " + df['review_content']
df['full_review'] = df['full_review'].apply(clean_text)

In [22]:
X = df['full_review'].values
y = np.round(df['rating'].values).astype(int)
y = y - 1

# Task 2: Tokenize and Encode

In [25]:
vocab_size = 10000
max_length = 150 # Max words per review
embedding_dim = 64

In [26]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

In [27]:
sequences = tokenizer.texts_to_sequences(X)
X_padded = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)

# Task 3 & 4: Model Building and Training

In [29]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(5, activation='softmax') # Changed to 5 output nodes with softmax
])

In [30]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

In [31]:
history = model.fit(
    X_train, y_train,
    epochs=15, # Increased slightly to allow classes to separate
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.9334 - loss: 0.6300 - val_accuracy: 0.9556 - val_loss: 0.2187
Epoch 2/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9496 - loss: 0.2962 - val_accuracy: 0.9556 - val_loss: 0.2189
Epoch 3/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9505 - loss: 0.2671 - val_accuracy: 0.9556 - val_loss: 0.2183
Epoch 4/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9505 - loss: 0.2673 - val_accuracy: 0.9556 - val_loss: 0.2143
Epoch 5/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9505 - loss: 0.2738 - val_accuracy: 0.9556 - val_loss: 0.2141
Epoch 6/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9505 - loss: 0.2629 - val_accuracy: 0.9556 - val_loss: 0.2124
Epoch 7/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9505 - loss: 0.2420 - val_accuracy: 0.9556 - val_loss: 0.2173
Epoch 8/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9505 - loss: 0.2021 - val_accuracy: 0.9556 - v

In [33]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nModel Evaluation -> Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")


Model Evaluation -> Test Loss: 0.4153, Test Accuracy: 0.9522


# Task 5: Compare and Demonstrate Predictions

In [34]:
print("\nComparing Actual vs Predicted Ratings on Test Set:")
predictions_probs = model.predict(X_test)


Comparing Actual vs Predicted Ratings on Test Set:
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step


In [35]:
# Get the index of the highest probability class
predictions = np.argmax(predictions_probs, axis=1)

In [36]:
for i in range(5):
    # Shift back from 0-4 to 1-5 scale for readability
    actual = y_test[i] + 1
    pred = predictions[i] + 1
    print(f"Actual Rating: {actual} Stars | Predicted Rating: {pred} Stars")

Actual Rating: 4 Stars | Predicted Rating: 4 Stars
Actual Rating: 4 Stars | Predicted Rating: 4 Stars
Actual Rating: 3 Stars | Predicted Rating: 4 Stars
Actual Rating: 4 Stars | Predicted Rating: 4 Stars
Actual Rating: 4 Stars | Predicted Rating: 4 Stars


In [41]:
print("\nDemonstrating prediction on unseen reviews...")
unseen_reviews = [
    "Not interested! The quality is poor and it is not working exactly as described.",
    "Terrible experience. The item broke after two days and the customer service was awful.",
    "It is okay. Not the best quality, but it gets the job done for the price."
]


Demonstrating prediction on unseen reviews...


In [42]:
cleaned_unseen = [clean_text(r) for r in unseen_reviews]
unseen_seq = tokenizer.texts_to_sequences(cleaned_unseen)
unseen_padded = pad_sequences(unseen_seq, maxlen=max_length, padding='post', truncating='post')

In [43]:
unseen_probs = model.predict(unseen_padded)
unseen_preds = np.argmax(unseen_probs, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


In [44]:
for review, pred in zip(unseen_reviews, unseen_preds):
    final_pred = pred + 1 # Shift back to 1-5 scale
    print(f"\nReview: '{review}'\nPredicted Rating: {final_pred} Stars")


Review: 'Not interested! The quality is poor and it is not working exactly as described.'
Predicted Rating: 4 Stars

Review: 'Terrible experience. The item broke after two days and the customer service was awful.'
Predicted Rating: 4 Stars

Review: 'It is okay. Not the best quality, but it gets the job done for the price.'
Predicted Rating: 4 Stars
